# 面试问题：ROC-AUC、PR-AUC、F1 应怎样从零计算，业务阈值如何选择？

**一句话回答**：先按 score 降序稳定排序，只在唯一 score 边界更新 TP/FP。ROC 画 TPR-FPR，AUC 衡量随机正例排在随机负例前的概率；PR 画 precision-recall，在稀有正类时更能反映误报。F1/Youden/成本最优阈值回答不同业务目标，阈值只能在 validation 选，test 负责一次性验证。

本 Notebook 用 NumPy 实现曲线、梯形积分、Average Precision、ties、成本阈值和 group bootstrap。

In [ ]:
import hashlib, json, math
import numpy as np

SEED93=9301; rng93=np.random.default_rng(SEED93)
n93=1600; y93=(rng93.random(n93)<.12).astype(int); latent93=rng93.normal(size=n93)+1.5*y93; score93=1/(1+np.exp(-latent93)); groups93=np.repeat(np.arange(400),4)
assert len(y93)==len(score93)==len(groups93)==1600
assert set(y93)=={0,1} and np.all((score93>0)&(score93<1))
assert len(np.unique(groups93))==400

## 1. 混淆矩阵是所有阈值指标的地基

规定 `score >= threshold` 为正。TPR/recall=`TP/P`，FPR=`FP/N`，precision=`TP/(TP+FP)`，specificity=`TN/N`。没有正类/负类时 ROC-AUC 无定义，不能返回一个看似正常的 0.5 掩盖数据问题。

阈值边界是否包含等号必须固定，否则 ties 会出现不一致。

In [ ]:
def confusion93(y,score,t):
    y=np.asarray(y,int); pred=np.asarray(score)>=t; return int(np.sum(pred&(y==1))),int(np.sum(pred&(y==0))),int(np.sum(~pred&(y==1))),int(np.sum(~pred&(y==0)))
cm93=confusion93([1,0,1,0],[.9,.8,.5,.1],.5)
assert cm93==(2,1,0,1)
assert confusion93([1,0],[.5,.5],.5)==(1,1,0,0)
assert sum(confusion93(y93,score93,.5))==n93

## 2. 手写 ROC 曲线

按 score 降序排列；同分样本必须作为一组一起加入，避免 ties 内部任意顺序改变曲线。加入起点 `(0,0)` 和终点 `(1,1)`，用 trapezoid 积分。AUC 对任何严格单调 score 变换不变，但不反映概率校准和具体阈值成本。

这里返回 threshold、FPR、TPR 三列。

In [ ]:
def roc_curve93(y,score):
    y=np.asarray(y,int); s=np.asarray(score,float)
    if len(y)!=len(s) or set(np.unique(y))!={0,1} or not np.isfinite(s).all(): raise ValueError("roc_contract")
    order=np.argsort(-s,kind="stable"); y=y[order]; s=s[order]; P=y.sum(); N=len(y)-P; rows=[(math.inf,0.,0.)]; tp=fp=0; i=0
    while i<len(y):
        value=s[i]; j=i
        while j<len(y) and s[j]==value: tp+=int(y[j]==1); fp+=int(y[j]==0); j+=1
        rows.append((float(value),fp/N,tp/P)); i=j
    return np.array(rows,float)
def trapz93(x,y): return float(np.sum((x[1:]-x[:-1])*(y[1:]+y[:-1])/2))
roc93=roc_curve93(y93,score93); auc93=trapz93(roc93[:,1],roc93[:,2])
assert np.all(np.diff(roc93[:,1])>=0) and np.all(np.diff(roc93[:,2])>=0)
assert .75<auc93<.9 and roc93[0,1:].tolist()==[0.,0.]
assert math.isclose(auc93,trapz93(roc_curve93(y93,score93**3)[:,1],roc_curve93(y93,score93**3)[:,2]),abs_tol=1e-12)

## 3. AUC 的排序概率解释

AUC 等于 `P(score_positive > score_negative)+0.5 P(tie)`。这个 Mann-Whitney 计算是独立 oracle，能抓出 ROC 实现的排序/tie bug。若交换正负 score，AUC 接近 `1-AUC`。

AUC 很高仍可能在业务要求的低 FPR 区域表现差，因此还应报告 partial region 或具体 operating point。

In [ ]:
def pair_auc93(y,score):
    pos=np.asarray(score)[np.asarray(y)==1]; neg=np.asarray(score)[np.asarray(y)==0]
    if not len(pos) or not len(neg): raise ValueError("auc_single_class")
    return float(np.mean((pos[:,None]>neg[None,:])+.5*(pos[:,None]==neg[None,:])))
pair93=pair_auc93(y93,score93)
assert math.isclose(pair93,auc93,abs_tol=1e-12)
assert pair_auc93([1,0],[.5,.5])==.5
try: pair_auc93([1,1],[.2,.8]); raise AssertionError("single class accepted")
except ValueError as e: assert str(e)=="auc_single_class"

## 4. PR 曲线与 Average Precision

PR 从高分向低分累积，precision=`TP/(TP+FP)`，recall=`TP/P`。Average Precision 用每次 recall 增量乘该点 precision，是阶梯积分；不要随意用梯形 PR-AUC 与 AP 混称。随机模型 AP 基线约等于阳性率，因此跨先验数据不可直接比较。

同样在唯一 score 边界更新。

In [ ]:
def pr_curve93(y,score):
    y=np.asarray(y,int); s=np.asarray(score,float); P=y.sum()
    if P==0 or P==len(y): raise ValueError("pr_single_class")
    order=np.argsort(-s,kind="stable"); y=y[order]; s=s[order]; rows=[(math.inf,0.,1.)]; tp=fp=0; i=0
    while i<len(y):
        value=s[i]; j=i
        while j<len(y) and s[j]==value: tp+=int(y[j]==1); fp+=int(y[j]==0); j+=1
        rows.append((float(value),tp/P,tp/(tp+fp))); i=j
    return np.array(rows,float)
pr93=pr_curve93(y93,score93); ap93=float(np.sum(np.diff(pr93[:,1])*pr93[1:,2]))
assert np.all(np.diff(pr93[:,1])>=0) and 0<=ap93<=1
assert ap93>y93.mean()*2
assert math.isclose(pr93[-1,2],y93.mean()) and math.isclose(pr93[-1,1],1.)

## 5. F1、Youden 和成本阈值不是一回事

F1 忽略 TN，适合需要平衡 precision/recall 的场景；Youden=`TPR-FPR` 给 sensitivity/specificity 同权；业务成本最小化显式给 FN/FP 价格。阈值还可受人工审核容量或最低 recall 约束。

下面在 validation 上同时计算三种选择，证明它们通常不同。

In [ ]:
val93=np.arange(0,800); test93=np.arange(800,1600)
def threshold_table93(y,score,fn_cost=7,fp_cost=1):
    out=[]
    for t in np.unique(np.r_[0.,score,1.]):
        tp,fp,fn,tn=confusion93(y,score,t); precision=tp/(tp+fp) if tp+fp else 0; recall=tp/(tp+fn) if tp+fn else 0; fpr=fp/(fp+tn) if fp+tn else 0; f1=2*precision*recall/(precision+recall) if precision+recall else 0; out.append({"t":float(t),"f1":f1,"youden":recall-fpr,"cost":fn_cost*fn+fp_cost*fp})
    return out
table93=threshold_table93(y93[val93],score93[val93]); t_f193=max(table93,key=lambda r:(r["f1"],-r["t"]))["t"]; t_youden93=max(table93,key=lambda r:(r["youden"],-r["t"]))["t"]; t_cost93=min(table93,key=lambda r:(r["cost"],r["t"]))["t"]
assert all(0<=t<=1 for t in (t_f193,t_youden93,t_cost93))
assert len({round(t,6) for t in (t_f193,t_youden93,t_cost93)})>=2
assert min(r["cost"] for r in table93)<=7*int(y93[val93].sum())

## 6. 封存 test 与 operating point

validation 选出阈值后，只在 test 计算一次混淆矩阵和区间；不能看到 test 结果后改成本或阈值。线上先验/成本变化时可以更新阈值，但要使用新 validation/实验数据并产生新版本。

同时报告 test ROC-AUC/AP，它们评价排序；阈值指标评价决策。

In [ ]:
test_cm93=confusion93(y93[test93],score93[test93],t_cost93); tp93,fp93,fn93,tn93=test_cm93; test_precision93=tp93/(tp93+fp93); test_recall93=tp93/(tp93+fn93); test_auc93=trapz93(roc_curve93(y93[test93],score93[test93])[:,1],roc_curve93(y93[test93],score93[test93])[:,2]); test_pr93=pr_curve93(y93[test93],score93[test93]); test_ap93=float(np.sum(np.diff(test_pr93[:,1])*test_pr93[1:,2]))
assert sum(test_cm93)==len(test93)
assert 0<=test_precision93<=1 and 0<=test_recall93<=1
assert .7<test_auc93<.95 and test_ap93>y93[test93].mean()

## 7. Group-level bootstrap

同一用户多行相关，必须按随机化/独立单位 group 重采样。这里每 group 4 行，bootstrap test AUC；单行 bootstrap 会低估方差。无正/负类的 resample 跳过并记录。

区间用于表达抽样不确定性，不修复数据泄漏或分布漂移。

In [ ]:
test_groups93=np.unique(groups93[test93]); boot_rng93=np.random.default_rng(9310); boot_auc93=[]
for _ in range(300):
    chosen=boot_rng93.choice(test_groups93,len(test_groups93),replace=True); idx=np.concatenate([test93[groups93[test93]==g] for g in chosen]);
    if len(np.unique(y93[idx]))==2: boot_auc93.append(pair_auc93(y93[idx],score93[idx]))
ci93=np.quantile(boot_auc93,[.025,.975])
assert len(boot_auc93)>290 and ci93[0]<test_auc93<ci93[1]
assert 0<ci93[0]<ci93[1]<1
assert ci93[1]-ci93[0]>.01

## 8. 指标合同与防错

manifest 绑定正类语义、score 方向、tie/threshold 约定、split、成本、阈值和实现版本。常见 bug：把较小 score 当更正、拿 hard label 算 AUC、micro/macro 混淆、空类别静默返回、在 test 调阈值。

线上 dashboard 同时展示排序指标、operating point、绝对量、先验和分组结果。

In [ ]:
manifest93={"schema":1,"positive_label":1,"score_direction":"higher_is_positive","threshold_rule":"score_gte","auc_ties":"half_credit","threshold":float(t_cost93),"selection":"validation_min_7fn_plus_fp","split":"v1"}; digest93=hashlib.sha256(json.dumps(manifest93,sort_keys=True,separators=(",",":")).encode()).hexdigest()
assert len(digest93)==64 and manifest93["threshold_rule"]=="score_gte"
assert math.isclose(manifest93["threshold"],t_cost93)
assert manifest93["positive_label"] in set(y93)
print({"roc_auc":round(test_auc93,3),"ap":round(test_ap93,3),"threshold":round(t_cost93,3),"precision":round(test_precision93,3),"recall":round(test_recall93,3)})

## 9. 面试收束、参考与练习

回答闭环：混淆矩阵 → 稳定 ties → ROC/AUC 与排序概率 → PR/AP 与先验 → 三种阈值目标 → validation/test 隔离 → group bootstrap → 指标版本。AUC 不是一个脱离业务阈值的万能分数。

练习：实现 partial AUC；加入 minimum recall/审核容量约束；实现 multilabel micro/macro AP；构造 ROC 高但低 FPR 很差的反例。

参考：[ROC 分析经典论文](https://doi.org/10.1016/j.patrec.2005.10.010)、[PR 与 ROC 的关系](https://dl.acm.org/doi/10.1145/1143844.1143874)、[Average Precision 定义参考](https://scikit-learn.org/stable/modules/model_evaluation.html#precision-recall-f-measure-metrics)。计算核心均已手写。